# Undiagnosed Hypertension and Diabetes Screening with NHANES
## Part 5: Recalibration and a paper-based point score

Upload `nhanes_analytic.csv` (saved by Part 1) when the first cell asks for it. Runtime is about 5 to 10 minutes.

**What this part does**
1. **Recalibration.** Fits the models on 2011 to 2016 only, updates their calibration on 2017 to 2020
   (intercept only, or intercept and slope) and checks on 2021 to 2023 whether the observed to expected
   (O/E) ratio moves toward 1, overall and in each subgroup.
2. **Local recalibration.** Simulates a clinic that recalibrates the final model on a small local sample:
   pilot samples of 250, 500, 1,000 and 2,000 people are drawn from 2021 to 2023, the intercept is updated
   on the pilot, and calibration is checked on the remaining participants (200 repeats per size).
3. **Point score.** Turns a logistic regression on four items (age, sex, waist-to-height ratio and systolic
   blood pressure) into whole-number points a nurse can add up on paper. Developed on 2011 to 2016, cut-off
   chosen on 2017 to 2020, tested once on 2021 to 2023.

### Step 5.1: Setup and data

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=False)

import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
SEED = 2026
np.random.seed(SEED)
OUT = "/content/part5_out" if os.path.isdir("/content") else "./part5_out"
os.makedirs(OUT, exist_ok=True)

CSV = "/content/nhanes_analytic.csv" if os.path.isdir("/content") else "nhanes_analytic.csv"
if not os.path.exists(CSV):
    from google.colab import files
    print("Upload nhanes_analytic.csv from Part 1")
    up = files.upload()
    CSV = "/content/" + list(up.keys())[0]
data = pd.read_csv(CSV)
print(data.shape)

### Step 5.2: Predictor sets and models (the same as Part 2, with the settings selected there)

In [ ]:
data["hx_htn"] = data["told_htn"]
data["hx_dm"] = (data["told_dm"] == 1).astype(float)
data.loc[data["told_dm"].isna(), "hx_dm"] = np.nan
data["smoking"] = data["smoking"].fillna("missing")

NUM_DM = ["age", "bmi", "waist_cm", "height_cm", "whtr", "sbp_mean", "dbp_mean", "sedentary_min"]
BIN_DM = ["sex_female", "hx_htn", "cvd_history", "active_leisure"]
CAT_DM = ["smoking", "education"]
NUM_HTN = ["age", "bmi", "waist_cm", "height_cm", "whtr", "sedentary_min"]
BIN_HTN = ["sex_female", "hx_dm", "cvd_history", "active_leisure"]
CAT_HTN = ["smoking", "education"]
TASKS = {
    "diabetes": {"pop": "dm_at_risk_pop", "y": "undx_dm", "num": NUM_DM, "bin": BIN_DM, "cat": CAT_DM},
    "hypertension": {"pop": "htn_at_risk_pop", "y": "undx_htn", "num": NUM_HTN, "bin": BIN_HTN, "cat": CAT_HTN},
}
# Settings selected on 2017-2020 in Part 2
BEST = {("diabetes", "LR"): {"C": 1.0}, ("hypertension", "LR"): {"C": 0.1},
        ("diabetes", "XGBoost"): {"n": 200, "depth": 2, "lr": 0.03},
        ("hypertension", "XGBoost"): {"n": 200, "depth": 3, "lr": 0.03}}

def task_data(task):
    t = TASKS[task]
    d = data[(data[t["pop"]] == 1) & data[t["y"]].notna()].copy()
    d["education"] = d["education"].map(lambda v: "missing" if pd.isna(v) else str(int(v)))
    return d

def preprocessor(num, bin_, cat):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]), num),
        ("bin", SimpleImputer(strategy="most_frequent"), bin_),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", drop="first"))]), cat),
    ])

def tree_matrix(d, num, bin_, cat):
    X = d[num + bin_].copy()
    for c in cat:
        X = pd.concat([X, pd.get_dummies(d[c], prefix=c, dtype=float)], axis=1)
    X.columns = [re.sub(r"[^0-9a-zA-Z_]", "_", str(c)) for c in X.columns]
    return X

def fit_predict(name, params, tr, tes, t):
    """Fit on tr and return predictions for each data frame in tes."""
    num, bin_, cat, y = t["num"], t["bin"], t["cat"], t["y"]
    if name == "LR":
        m = Pipeline([("pre", preprocessor(num, bin_, cat)), ("m", LogisticRegression(max_iter=2000, C=params["C"]))])
        cols = num + bin_ + cat
        m.fit(tr[cols], tr[y])
        return [m.predict_proba(te[cols])[:, 1] for te in tes]
    m = XGBClassifier(n_estimators=params["n"], max_depth=params["depth"], learning_rate=params["lr"],
                      subsample=0.8, colsample_bytree=0.8, min_child_weight=5, eval_metric="logloss",
                      random_state=SEED, n_jobs=-1)
    Xtr = tree_matrix(tr, num, bin_, cat)
    m.fit(Xtr, tr[y])
    return [m.predict_proba(tree_matrix(te, num, bin_, cat).reindex(columns=Xtr.columns, fill_value=0))[:, 1] for te in tes]

### Step 5.3: Calibration helpers

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def expit(x):
    return 1 / (1 + np.exp(-x))

def cal_int_slope(y, p):
    lp = logit(p)
    slope = sm.GLM(y, sm.add_constant(lp), family=sm.families.Binomial()).fit().params[1]
    inter = sm.GLM(y, np.ones_like(lp), family=sm.families.Binomial(), offset=lp).fit().params[0]
    return inter, slope

def recal_intercept(y, p):
    """Calibration-in-the-large update: logit(p_new) = a + logit(p)."""
    a = sm.GLM(y, np.ones(len(p)), family=sm.families.Binomial(), offset=logit(p)).fit().params[0]
    return lambda q: expit(a + logit(q))

def recal_logistic(y, p):
    """Logistic recalibration: logit(p_new) = a + b * logit(p)."""
    a, b = sm.GLM(y, sm.add_constant(logit(p)), family=sm.families.Binomial()).fit().params
    return lambda q: expit(a + b * logit(q))

RACE = {1: "Mexican American", 2: "Other Hispanic", 3: "Non-Hispanic White", 4: "Non-Hispanic Black",
        6: "Non-Hispanic Asian", 7: "Other or multiracial"}
def subgroups(d):
    g = {"Sex: male": d["sex_female"] == 0, "Sex: female": d["sex_female"] == 1,
         "Age 20-39": d["age"] < 40, "Age 40-59": (d["age"] >= 40) & (d["age"] < 60), "Age 60+": d["age"] >= 60}
    for k, v in RACE.items():
        g[f"Race: {v}"] = d["race_eth"] == k
    g["Income: below 1.3 x poverty"] = d["pir"] < 1.3
    g["Income: 1.3 to 3.5"] = (d["pir"] >= 1.3) & (d["pir"] < 3.5)
    g["Income: 3.5 or more"] = d["pir"] >= 3.5
    return g

def oe(y, p):
    return np.sum(y) / np.sum(p)

### Step 5.4: Recalibration on 2017-2020, tested on 2021-2023
Models are fitted on 2011 to 2016 only, so the 2017 to 2020 data used for recalibration are new to them.
The final models from Part 2 (fitted on 2011 to 2020, no recalibration) are shown for comparison.

In [ ]:
cal_rows, sub_rows = [], []
for task, t in TASKS.items():
    d = task_data(task)
    tr, va, te = d[d.split == "train"], d[d.split == "validation"], d[d.split == "test"]
    trva = pd.concat([tr, va])
    yv, yt = va[t["y"]].values, te[t["y"]].values
    for name in ["LR", "XGBoost"]:
        params = BEST[(task, name)]
        pv, pt = fit_predict(name, params, tr, [va, te], t)
        (pt_final,) = fit_predict(name, params, trva, [te], t)
        versions = {
            "Final model (2011-2020, no update)": pt_final,
            "Fitted on 2011-2016, no update": pt,
            "Intercept updated on 2017-2020": recal_intercept(yv, pv)(pt),
            "Intercept and slope updated on 2017-2020": recal_logistic(yv, pv)(pt),
        }
        for label, p in versions.items():
            inter, slope = cal_int_slope(yt, p)
            ratios = {}
            for gname, mask in subgroups(te).items():
                mk = mask.values
                if yt[mk].sum() >= 10:
                    ratios[gname] = oe(yt[mk], p[mk])
                sub_rows.append({"task": task, "model": name, "version": label, "subgroup": gname,
                                 "n": int(mk.sum()), "cases": int(yt[mk].sum()),
                                 "observed_%": 100 * yt[mk].mean(), "expected_%": 100 * p[mk].mean(),
                                 "O/E": oe(yt[mk], p[mk])})
            spread = np.abs(np.log(list(ratios.values())))
            cal_rows.append({"task": task, "model": name, "version": label, "AUC": roc_auc_score(yt, p),
                             "Brier": brier_score_loss(yt, p), "O/E": oe(yt, p), "Cal_intercept": inter,
                             "Cal_slope": slope, "Subgroup_O/E_min": min(ratios.values()),
                             "Subgroup_O/E_max": max(ratios.values()),
                             "Mean_abs_log_subgroup_O/E": spread.mean()})
cal = pd.DataFrame(cal_rows).round(3)
subs = pd.DataFrame(sub_rows).round(3)
cal.to_csv(os.path.join(OUT, "recalibration_test.csv"), index=False)
subs.to_csv(os.path.join(OUT, "recalibration_subgroups.csv"), index=False)
print(cal.to_string(index=False))

### Step 5.5: Local recalibration on a small pilot sample
The final model (fitted on 2011 to 2020) is recalibrated on a random pilot sample from 2021 to 2023 and
evaluated on everyone else in 2021 to 2023. This mimics a clinic that collects a few hundred local records,
including HbA1c or blood pressure, before using the model.

In [ ]:
rng = np.random.default_rng(SEED)
pilot_rows = []
for task, t in TASKS.items():
    d = task_data(task)
    tr, va, te = d[d.split == "train"], d[d.split == "validation"], d[d.split == "test"]
    trva = pd.concat([tr, va])
    yt = te[t["y"]].values
    for name in ["LR", "XGBoost"]:
        (pt,) = fit_predict(name, BEST[(task, name)], trva, [te], t)
        for n_pilot in [250, 500, 1000, 2000]:
            for rep in range(200):
                idx = rng.permutation(len(te))
                pi, ev = idx[:n_pilot], idx[n_pilot:]
                if yt[pi].sum() == 0:
                    pilot_rows.append({"task": task, "model": name, "n_pilot": n_pilot, "rep": rep,
                                       "pilot_cases": 0, "O/E_before": oe(yt[ev], pt[ev]), "O/E_after": np.nan,
                                       "slope_after": np.nan})
                    continue
                f = recal_intercept(yt[pi], pt[pi])
                p_ev = f(pt[ev])
                pilot_rows.append({"task": task, "model": name, "n_pilot": n_pilot, "rep": rep,
                                   "pilot_cases": int(yt[pi].sum()), "O/E_before": oe(yt[ev], pt[ev]),
                                   "O/E_after": oe(yt[ev], p_ev)})
pilot = pd.DataFrame(pilot_rows)
pilot.to_csv(os.path.join(OUT, "local_recalibration_repeats.csv"), index=False)
summ = (pilot.groupby(["task", "model", "n_pilot"])
        .agg(median_pilot_cases=("pilot_cases", "median"), OE_before=("O/E_before", "median"),
             OE_after_median=("O/E_after", "median"),
             OE_after_2_5=("O/E_after", lambda s: s.quantile(0.025)),
             OE_after_97_5=("O/E_after", lambda s: s.quantile(0.975)),
             share_within_0_8_to_1_25=("O/E_after", lambda s: ((s >= 0.8) & (s <= 1.25)).mean()),
             failed=("O/E_after", lambda s: s.isna().mean()))
        .reset_index().round(3))
summ.to_csv(os.path.join(OUT, "local_recalibration_summary.csv"), index=False)
print(summ.to_string(index=False))

### Step 5.6: A paper-based point score for undiagnosed diabetes
The score uses only items that need a tape measure, a stadiometer and a blood pressure cuff. Each item is
grouped into bands, a logistic regression is fitted on 2011 to 2016, and each coefficient is divided by the
coefficient for male sex and rounded, so one point equals the effect of being male. A missing measurement
scores 0 points. Three candidate item sets are compared on 2017 to 2020 and the one with the highest
validation AUC is kept; the test data play no part in this choice.

In [ ]:
AGE = ([0, 40, 50, 60, 70, 200], ["20-39", "40-49", "50-59", "60-69", "70+"])
SBP = ([0, 120, 140, 160, 400], ["<120", "120-139", "140-159", ">=160"])
CANDIDATES = {
    "Age, sex, WHtR (4 bands), SBP": {"age": AGE, "whtr": ([0, 0.5, 0.6, 0.7, 10], ["<0.50", "0.50-0.59", "0.60-0.69", ">=0.70"]), "sbp_mean": SBP},
    "Age, sex, WHtR (6 bands), SBP": {"age": AGE, "whtr": ([0, 0.5, 0.55, 0.6, 0.65, 0.7, 10], ["<0.50", "0.50-0.54", "0.55-0.59", "0.60-0.64", "0.65-0.69", ">=0.70"]), "sbp_mean": SBP},
    "Age, sex, WHtR (4 bands), BMI, SBP": {"age": AGE, "whtr": ([0, 0.5, 0.6, 0.7, 10], ["<0.50", "0.50-0.59", "0.60-0.69", ">=0.70"]),
                                           "bmi": ([0, 25, 30, 35, 100], ["<25", "25-29.9", "30-34.9", ">=35"]), "sbp_mean": SBP},
}

def banded(d, bands):
    X = pd.DataFrame(index=d.index)
    for v, (cuts, labels) in bands.items():
        b = pd.cut(d[v], cuts, right=False, labels=labels)
        for lab in labels[1:]:
            X[f"{v}={lab}"] = (b == lab).astype(float)
    X["male=yes"] = (d["sex_female"] == 0).astype(float)
    return X

def make_score(bands, d_fit):
    X = banded(d_fit, bands)
    fit = sm.GLM(d_fit["undx_dm"].values, sm.add_constant(X), family=sm.families.Binomial()).fit()
    beta = fit.params.drop("const")
    pts = (beta / beta["male=yes"]).round().astype(int)
    return beta, pts, (lambda d: banded(d, bands)[pts.index].values @ pts.values)

dm = task_data("diabetes")
tr, va, te = dm[dm.split == "train"], dm[dm.split == "validation"], dm[dm.split == "test"]
cand_rows = []
for cname, bands in CANDIDATES.items():
    beta, pts, f = make_score(bands, tr)
    cand_rows.append({"candidate": cname, "validation_AUC": roc_auc_score(va["undx_dm"], f(va)),
                      "max_points": int(pts.groupby(lambda k: k.split("=")[0]).max().clip(lower=0).sum())})
cands = pd.DataFrame(cand_rows).round(3)
cands.to_csv(os.path.join(OUT, "point_score_candidates.csv"), index=False)
print(cands.to_string(index=False))
chosen = cands.sort_values("validation_AUC", ascending=False).iloc[0]["candidate"]
BANDS = CANDIDATES[chosen]
beta, points, score = make_score(BANDS, tr)
coef_table = pd.DataFrame({"coefficient": beta.round(3), "odds_ratio": np.exp(beta).round(2), "points": points})
coef_table.to_csv(os.path.join(OUT, "point_score_items.csv"))
print("Chosen:", chosen)
print(coef_table.to_string())
s_tr, s_va, s_te = score(tr), score(va), score(te)
print("Score range:", s_tr.min(), "to", s_tr.max())

### Step 5.7: Cut-off, risk table and test performance of the point score

In [ ]:
yv, yt = va["undx_dm"].values, te["undx_dm"].values
# cut-off: the highest score that still reaches 80% sensitivity on 2017-2020
cut = max(c for c in range(int(s_va.max()) + 1) if (s_va[yv == 1] >= c).mean() >= 0.80)
def perf(y, s, c):
    f = s >= c
    tp, fp, fn, tn = (f & (y == 1)).sum(), (f & (y == 0)).sum(), (~f & (y == 1)).sum(), (~f & (y == 0)).sum()
    return {"Sensitivity": tp / (tp + fn), "Specificity": tn / (tn + fp), "PPV": tp / (tp + fp), "Flagged_%": 100 * f.mean()}

def boot_auc(y, p, n=1000):
    r = np.random.default_rng(SEED); a = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            a.append(roc_auc_score(y[i], p[i]))
    return np.percentile(a, [2.5, 97.5])

(p_lr,) = fit_predict("LR", BEST[("diabetes", "LR")], pd.concat([tr, va]), [te], TASKS["diabetes"])
lo, hi = boot_auc(yt, s_te)
lo2, hi2 = boot_auc(yt, p_lr)
d_auc = []
r = np.random.default_rng(SEED)
for _ in range(1000):
    i = r.integers(0, len(yt), len(yt))
    d_auc.append(roc_auc_score(yt[i], p_lr[i]) - roc_auc_score(yt[i], s_te[i]))
score_perf = pd.DataFrame([
    {"model": f"Point score (cut-off {cut} points)", "AUC": roc_auc_score(yt, s_te), "AUC_lo": lo, "AUC_hi": hi, **perf(yt, s_te, cut)},
    {"model": f"Point score (cut-off {cut + 1} points)", "AUC": roc_auc_score(yt, s_te), "AUC_lo": lo, "AUC_hi": hi, **perf(yt, s_te, cut + 1)},
    {"model": "Full logistic regression (Part 2)", "AUC": roc_auc_score(yt, p_lr), "AUC_lo": lo2, "AUC_hi": hi2,
     **perf(yt, p_lr, np.quantile(p_lr, 1 - perf(yt, s_te, cut)["Flagged_%"] / 100))},
]).round(3)
score_perf["AUC_difference_LR_minus_score"] = [np.nan, np.nan, round(roc_auc_score(yt, p_lr) - roc_auc_score(yt, s_te), 3)]
score_perf["difference_CI"] = [np.nan, np.nan, f"{np.percentile(d_auc, 2.5):.3f} to {np.percentile(d_auc, 97.5):.3f}"]
print("Validation sensitivity at cut-offs", {c: round((s_va[yv == 1] >= c).mean(), 3) for c in range(int(s_va.max()) + 1)})
score_perf.to_csv(os.path.join(OUT, "point_score_performance.csv"), index=False)
print(score_perf.to_string(index=False))

# risk table: observed risk by score band in development and test data
bands = pd.cut(np.r_[s_tr, s_te], [-1, 2, 4, 6, 8, 100], labels=["0-2", "3-4", "5-6", "7-8", "9+"])
rt = pd.DataFrame({"band": bands, "y": np.r_[tr["undx_dm"].values, yt],
                   "period": ["2011-2016"] * len(s_tr) + ["2021-2023"] * len(s_te)})
risk_table = (rt.groupby(["period", "band"], observed=True)["y"].agg(n="size", cases="sum", risk_pct="mean").reset_index())
risk_table["risk_pct"] = (100 * risk_table["risk_pct"]).round(1)
risk_table.to_csv(os.path.join(OUT, "point_score_risk_table.csv"), index=False)
print(risk_table.to_string(index=False))

# subgroup performance of the score in 2021-2023
sg = []
for gname, mask in subgroups(te).items():
    mk = mask.values
    row = {"subgroup": gname, "n": int(mk.sum()), "cases": int(yt[mk].sum())}
    if yt[mk].sum() >= 10 and (yt[mk] == 0).sum() >= 10:
        row["AUC"] = roc_auc_score(yt[mk], s_te[mk])
        row["Sensitivity"] = (s_te[mk][yt[mk] == 1] >= cut).mean()
    sg.append(row)
pd.DataFrame(sg).round(3).to_csv(os.path.join(OUT, "point_score_subgroups.csv"), index=False)
print(pd.DataFrame(sg).round(3).to_string(index=False))

### Step 5.8: Save and download

In [ ]:
print("Saved to", OUT, sorted(os.listdir(OUT)))
try:
    from google.colab import files
    import shutil
    shutil.make_archive("/content/part5_results", "zip", OUT)
    files.download("/content/part5_results.zip")
except Exception:
    pass